In [5]:
import pandas as pd
import sqlite3
conn = sqlite3.connect(
    "../data/processed/credit_risk.db"
)

In [2]:
results = pd.read_csv(
    "../data/processed/final_credit_risk_results.csv"
)

results.head()

,ActualDefault,PD,RiskBand,LGD,EAD,ExpectedLoss
0,0,0.042467,Very Low,0.45,10000,191.099720
1,0,0.044836,Very Low,0.45,10000,201.760554
2,0,0.036785,Very Low,0.45,10000,165.534241
3,0,0.018944,Very Low,0.45,10000,85.246265
4,0,0.411452,Very High,0.45,10000,1851.533632


In [3]:
results.shape

(22460, 6)

In [4]:
conn = sqlite3.connect(
    "../data/processed/credit_risk.db"
)

In [5]:
results.to_sql(
    "credit_risk_results",
    conn,
    if_exists="replace",
    index=False
)

22460

In [6]:
query = """
SELECT *
FROM credit_risk_results
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,ActualDefault,PD,RiskBand,LGD,EAD,ExpectedLoss
0,0,0.042467,Very Low,0.45,10000,191.099720
1,0,0.044836,Very Low,0.45,10000,201.760554
2,0,0.036785,Very Low,0.45,10000,165.534241
3,0,0.018944,Very Low,0.45,10000,85.246265
4,0,0.411452,Very High,0.45,10000,1851.533632
5,1,0.083614,Low,0.45,10000,376.264269
6,0,0.037909,Very Low,0.45,10000,170.588912
7,0,0.092388,Low,0.45,10000,415.747003
8,0,0.010483,Very Low,0.45,10000,47.174448
9,0,0.041481,Very Low,0.45,10000,186.662891


In [7]:
query = """
SELECT
    COUNT(*) AS borrowers,
    AVG(PD) AS average_pd,
    AVG(ActualDefault) AS actual_default_rate,
    SUM(EAD) AS total_ead,
    SUM(ExpectedLoss) AS total_expected_loss
FROM credit_risk_results;
"""

portfolio_summary = pd.read_sql_query(
    query,
    conn
)

portfolio_summary

,borrowers,average_pd,actual_default_rate,total_ead,total_expected_loss
0,22460,0.064691,0.065984,224600000,6.538286e+06


In [8]:
query = """
SELECT
    RiskBand,
    COUNT(*) AS borrowers,
    ROUND(AVG(PD), 4) AS average_pd,
    ROUND(AVG(ActualDefault), 4) AS actual_default_rate,
    ROUND(AVG(ExpectedLoss), 2) AS average_expected_loss,
    ROUND(SUM(ExpectedLoss), 2) AS total_expected_loss
FROM credit_risk_results
GROUP BY RiskBand
ORDER BY average_pd;
"""

risk_band_sql = pd.read_sql_query(
    query,
    conn
)

risk_band_sql

,RiskBand,borrowers,average_pd,actual_default_rate,average_expected_loss,total_expected_loss
0,Very Low,15679,0.0213,0.0193,95.68,1500198.43
1,Low,3606,0.0717,0.0837,322.76,1163883.01
2,Moderate,1675,0.1364,0.1510,614.02,1028491.36
3,High,840,0.2751,0.3202,1238.16,1040052.74
4,Very High,660,0.6080,0.5379,2735.85,1805660.71


In [9]:
query = """
SELECT
    COUNT(*) AS high_risk_borrowers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM credit_risk_results),
        2
    ) AS percentage_of_portfolio
FROM credit_risk_results
WHERE RiskBand IN ('High', 'Very High');
"""

high_risk_summary = pd.read_sql_query(
    query,
    conn
)

high_risk_summary

,high_risk_borrowers,percentage_of_portfolio
0,1500,6.68


In [10]:
query = """
SELECT
    RiskBand,
    COUNT(*) AS borrowers,
    ROUND(SUM(EAD), 2) AS total_ead,
    ROUND(SUM(ExpectedLoss), 2) AS expected_loss
FROM credit_risk_results
WHERE RiskBand IN ('High', 'Very High')
GROUP BY RiskBand
ORDER BY expected_loss DESC;
"""

pd.read_sql_query(
    query,
    conn
)

,RiskBand,borrowers,total_ead,expected_loss
0,Very High,660,6600000.0,1805660.71
1,High,840,8400000.0,1040052.74


In [11]:
query = """
SELECT
    PD,
    ActualDefault,
    RiskBand,
    ExpectedLoss
FROM credit_risk_results
ORDER BY PD DESC
LIMIT 10;
"""

pd.read_sql_query(
    query,
    conn
)

,PD,ActualDefault,RiskBand,ExpectedLoss
0,0.944662,1,Very High,4250.979371
1,0.942773,1,Very High,4242.480439
2,0.940079,1,Very High,4230.354966
3,0.939845,1,Very High,4229.300376
4,0.921605,1,Very High,4147.220769
5,0.917885,0,Very High,4130.484536
6,0.915992,1,Very High,4121.963139
7,0.912950,1,Very High,4108.275542
8,0.909614,1,Very High,4093.260985
9,0.906317,1,Very High,4078.424303


In [6]:
query = """
WITH risk_deciles AS (
    SELECT
        PD,
        ActualDefault,
        ExpectedLoss,
        NTILE(10) OVER (
            ORDER BY PD DESC
        ) AS risk_decile
    FROM credit_risk_results
)

SELECT
    risk_decile,
    COUNT(*) AS borrowers,
    ROUND(AVG(PD), 4) AS average_pd,
    ROUND(AVG(ActualDefault), 4) AS actual_default_rate,
    SUM(ActualDefault) AS actual_defaults,
    ROUND(SUM(ExpectedLoss), 2) AS expected_loss
FROM risk_deciles
GROUP BY risk_decile
ORDER BY risk_decile;
"""

decile_summary = pd.read_sql_query(
    query,
    conn
)

decile_summary

,risk_decile,borrowers,average_pd,actual_default_rate,actual_defaults,expected_loss
0,1,2246,0.3359,0.3353,753,3394785.79
1,2,2246,0.0989,0.1140,256,999679.29
2,3,2246,0.0627,0.0744,167,633913.56
3,4,2246,0.0424,0.0472,106,428593.91
4,5,2246,0.0307,0.0321,72,309840.33
5,6,2246,0.0233,0.0236,53,235270.21
6,7,2246,0.0186,0.0111,25,187551.31
7,8,2246,0.0148,0.0102,23,149764.27
8,9,2246,0.0115,0.0067,15,116510.76
9,10,2246,0.0082,0.0053,12,82376.83


In [8]:
query = """
WITH risk_deciles AS (
    SELECT
        PD,
        ActualDefault,
        ExpectedLoss,
        NTILE(10) OVER (
            ORDER BY PD DESC
        ) AS risk_decile
    FROM credit_risk_results
)

SELECT
    risk_decile,
    COUNT(*) AS borrowers,
    ROUND(AVG(PD), 4) AS average_pd,
    ROUND(AVG(ActualDefault), 4) AS actual_default_rate,
    SUM(ActualDefault) AS actual_defaults,
    ROUND(SUM(ExpectedLoss), 2) AS expected_loss
FROM risk_deciles
GROUP BY risk_decile
ORDER BY risk_decile;
"""

decile_summary = pd.read_sql_query(
    query,
    conn
)

decile_summary

,risk_decile,borrowers,average_pd,actual_default_rate,actual_defaults,expected_loss
0,1,2246,0.3359,0.3353,753,3394785.79
1,2,2246,0.0989,0.1140,256,999679.29
2,3,2246,0.0627,0.0744,167,633913.56
3,4,2246,0.0424,0.0472,106,428593.91
4,5,2246,0.0307,0.0321,72,309840.33
5,6,2246,0.0233,0.0236,53,235270.21
6,7,2246,0.0186,0.0111,25,187551.31
7,8,2246,0.0148,0.0102,23,149764.27
8,9,2246,0.0115,0.0067,15,116510.76
9,10,2246,0.0082,0.0053,12,82376.83


In [9]:
query = """
WITH risk_deciles AS (
    SELECT
        PD,
        ActualDefault,
        NTILE(10) OVER (
            ORDER BY PD DESC
        ) AS risk_decile
    FROM credit_risk_results
),

decile_defaults AS (
    SELECT
        risk_decile,
        COUNT(*) AS borrowers,
        SUM(ActualDefault) AS defaults
    FROM risk_deciles
    GROUP BY risk_decile
),

cumulative AS (
    SELECT
        risk_decile,
        borrowers,
        defaults,
        SUM(defaults) OVER (
            ORDER BY risk_decile
        ) AS cumulative_defaults
    FROM decile_defaults
)

SELECT
    risk_decile,
    borrowers,
    defaults,
    cumulative_defaults,
    ROUND(
        100.0 * cumulative_defaults /
        SUM(defaults) OVER (),
        2
    ) AS cumulative_default_capture_pct
FROM cumulative
ORDER BY risk_decile;
"""

default_capture = pd.read_sql_query(
    query,
    conn
)

default_capture

,risk_decile,borrowers,defaults,cumulative_defaults,cumulative_default_capture_pct
0,1,2246,753,753,50.81
1,2,2246,256,1009,68.08
2,3,2246,167,1176,79.35
3,4,2246,106,1282,86.50
4,5,2246,72,1354,91.36
5,6,2246,53,1407,94.94
6,7,2246,25,1432,96.63
7,8,2246,23,1455,98.18
8,9,2246,15,1470,99.19
9,10,2246,12,1482,100.00
